# Lesson 14 — Branching Rules

## Learning objectives

1. Understand pseudocost branching.
2. Implement strong branching {cite:p}`Achterberg2005` inside a hand-rolled B&B harness.
3. Recognize reliability branching as a hybrid.
4. Recognize the cost-per-node vs branching-quality trade-off and articulate it from data.

## 1. The branching problem

After solving the LP relaxation, several variables may be fractional. *Which* one to branch on is the **branching rule**. A bad rule can blow up the tree by orders of magnitude {cite:p}`Achterberg2005`.

## 2. Most fractional / Most infeasible

Pick the $x_j$ closest to half-integer. Cheap, but a poor predictor of *useful* branches.

## 3. Pseudocost branching

For each variable $x_j$, track historical *per-unit* changes in LP value when branched up ($\Phi_j^+$) and down ($\Phi_j^-$). Up-branching pushes $x_j$ from its LP value $x_j^{LP}$ to $\lceil x_j^{LP} \rceil$, a step of $1 - f_j$ where $f_j = x_j^{LP} - \lfloor x_j^{LP} \rfloor$; down-branching takes a step of $f_j$. The expected LP-value gains are therefore $\Phi_j^+(1 - f_j)$ on the up child and $\Phi_j^-\, f_j$ on the down child. Score:

$$\text{score}(j) = \max(\Phi_j^+ (1 - f_j), \epsilon) \cdot \max(\Phi_j^- f_j, \epsilon)$$

Pick the variable with the largest score (product, so a *bad* child is heavily penalised). Cheap; uses information from the whole tree {cite:p}`Achterberg2005`.

## 4. Strong branching

For each candidate $x_j$, *actually solve* both child LPs (limited iterations) and record the LP value gains. Pick the $j$ with the best (e.g., product of) gains.

Expensive — solving 2 LPs per candidate per node. But on hard MILPs, the branching quality more than pays for itself {cite:p}`Achterberg2005`.

## 5. Reliability branching

Hybrid: use pseudocosts, but switch to strong branching for variables whose pseudocost is "unreliable" (fewer than $\eta_{\text{rel}}$ observations in each direction). Strong-branching probes are themselves recorded as pseudocost observations, so a variable "graduates" to pseudocost scoring after a few probes. The default in modern solvers {cite:p}`Achterberg2005,Achterberg2007`.

It is also `discopt`'s default: the Rust `TreeManager` scores candidates with the pseudocost *product* rule above, probes unreliable candidates with limited strong branching (in the current source, `crates/discopt-core/src/bnb/branching.rs` and `milp_driver.rs`: reliability threshold 8, at most 8 probed candidates per node, and probing only while the tree is still small), and falls back to most-fractional for variables with no history at all.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# A 30-item knapsack whose values equal its weights (a subset-sum-like
# instance: the LP bound is nearly useless, so the tree has to do the work).
def build():
    m = do.Model("br_demo")
    n = 30
    rng = np.random.default_rng(1)
    w = rng.integers(10, 100, size=n)
    v = w.copy()
    W = int(w.sum() / 2)
    x = m.binary("x", shape=(n,))
    m.subject_to(w @ x <= W)
    m.maximize(v @ x)
    return m

# discopt 0.9 has no public option for choosing the branching rule. (The old
# `branching_policy=` argument was removed in 0.7, and 0.9 rejects unknown
# solve() options with a TypeError instead of silently ignoring them.)
#
# By default a pure MILP goes to HiGHS, with discopt checking the certificate
# HiGHS returns (`r.algorithm_route` says so). `nlp_solver="simplex"` selects
# discopt's own Rust branch-and-bound instead (warm-started dual simplex at
# every node, reliability branching). Node counts from the two engines measure
# different trees, so do not compare them as if they came from the same rule.
for label, kw in [("default (HiGHS route)", {}),
                  ("discopt B&B (nlp_solver='simplex')", {"nlp_solver": "simplex"})]:
    r = build().solve(**kw)
    print(f"{label:36s} nodes={r.node_count:5d}  time={r.wall_time:.3f}s  z*={r.objective:.0f}")
    print(f"{'':36s} route: {r.algorithm_route or 'discopt native'}")


## 6. Branching directions

For each branching, we choose **down first** (explore $x_j \le \lfloor x_j^{LP} \rfloor$ first) or **up first**. Default is to follow the LP relaxation's *rounding*: if $x_j^{LP} = 0.7$, try up first.

## 7. Plunging

A heuristic: after expanding a node, dive depth-first into one child until you hit infeasibility or integrality. Tends to find incumbents fast. Most modern solvers do this opportunistically.

## 8. Comparing rules yourself

Because the branching rule is internal to `discopt`, the exercises compare most-fractional, strong, pseudocost and reliability branching in a small B&B harness that you write yourself (LP relaxations via `scipy.optimize.linprog`). There every choice, including the branching direction, is yours to control. Use `discopt` as the reference optimum to check that your harness is correct.

## References
{cite:p}`Achterberg2005,Achterberg2007` are the canonical references for modern branching.